In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score, recall_score
from catboost import CatBoostClassifier, Pool
import os
import numpy as np
from pymatgen.core import Composition, Element

In [ ]:
data = pd.read_csv('./data/data.csv')

In [ ]:
positive = data[data['label'] == 1]
negative = data[data['label'] == 0]

In [ ]:
N_POSITIVE = len(positive) // 2
test_positive = positive.sample(n=N_POSITIVE, random_state=42)
remove_test_positive = positive.drop(test_positive.index).reset_index(drop=True)
test_positive = test_positive.reset_index(drop=True)

In [ ]:
pod_features = [
    'pearson',
    'space',
    'volume',
    'density',
    
    'band_gap',
    'efermi',
    'formation_energy_per_atom',
    
    'MagpieData mean NsValence',
    'MagpieData mean NpValence',
    'MagpieData mean NdValence',
    'MagpieData mean NfValence',
    
    'MagpieData mean NUnfilled',
    
    'MagpieData mean CovalentRadius',
    'MagpieData mean AtomicWeight',
    
    'MagpieData mean Electronegativity',
    'MagpieData mean GSmagmom',
    
    'MagpieData mean Row',
    'MagpieData mean Column'
]

In [ ]:
X_test_positive = test_positive[pod_features]
y_test_positive = test_positive['label'] 

In [ ]:
n_positive = remove_test_positive.shape[0]
n_negative = n_positive

In [ ]:
categorical_features = ['pearson'] if 'pearson' in remove_test_positive[pod_features].columns else []

In [ ]:
N_RUNS = 500
ITERATIONS = 300
LEARNING_RATE = 0.05
DEPTH = 8
LOSS_FUNCTION = 'Logloss'
EVAL_METRIC = 'AUC'
RANDOM_SEED = 0
VERBOSE = 0

recall_per_run = []
accuracy_per_run = []
f1_per_run = []
auroc_per_run = []
auprc_per_run = []
model_per_run = []

for run in range(0, N_RUNS):
    X_train_negative = negative.sample(n=n_negative, random_state=run)
    neg_for_test_pool = negative.drop(X_train_negative.index)
    X_test_negative = neg_for_test_pool.sample(n=len(X_test_positive), random_state=run)[pod_features]
    
    all_df = pd.concat([remove_test_positive, X_train_negative], axis=0).reset_index(drop=True)
    X_all = all_df[pod_features]
    y_all = all_df['label']
    
    train_pool = Pool(X_all, y_all, cat_features=categorical_features)
    
    model = CatBoostClassifier(
        iterations=ITERATIONS,
        learning_rate=LEARNING_RATE,
        depth=DEPTH,
        loss_function=LOSS_FUNCTION,
        eval_metric=EVAL_METRIC,
        random_seed=RANDOM_SEED,
        verbose=VERBOSE
    )
    model.fit(train_pool)
    model_per_run.append(model)
    
    test_positive_pool = Pool(X_test_positive, cat_features=categorical_features)
    test_negative_pool = Pool(X_test_negative, cat_features=categorical_features)
    
    y_pred_proba_positive = model.predict_proba(test_positive_pool)[:, 1]
    y_pred_proba_negative = model.predict_proba(test_negative_pool)[:, 1]
    
    y_pred_positive = model.predict(test_positive_pool)
    y_pred_negative = model.predict(test_negative_pool)
    
    y_true = np.concatenate([y_test_positive, np.zeros(len(X_test_negative))])
    y_pred = np.concatenate([y_pred_positive, y_pred_negative])
    y_pred_proba = np.concatenate([y_pred_proba_positive, y_pred_proba_negative])
    
    recall = recall_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auroc = roc_auc_score(y_true, y_pred_proba)
    auprc = average_precision_score(y_true, y_pred_proba)
    
    recall_per_run.append(recall)
    accuracy_per_run.append(accuracy)
    f1_per_run.append(f1)
    auroc_per_run.append(auroc)
    auprc_per_run.append(auprc)
    
    print(f"Run {run}: Recall={recall:.3f}, Accuracy={accuracy:.3f}, F1={f1:.3f}, AUROC={auroc:.3f}, AUPRC={auprc:.3f}")

In [ ]:
recall_df = pd.DataFrame({
    "run": range(1, len(recall_per_run) + 1),
    "recall": pd.Series(recall_per_run, dtype=float),
    "accuracy": pd.Series(accuracy_per_run, dtype=float),
    "f1": pd.Series(f1_per_run, dtype=float),
    "auroc": pd.Series(auroc_per_run, dtype=float),
    "auprc": pd.Series(auprc_per_run, dtype=float)
})

In [ ]:
recall_df

In [ ]:
arr = np.array(recall_per_run)
topk = 50
top_idx = arr.argsort()[-topk:][::-1]
top_runs = [i + 1 for i in top_idx]

top_models = [model_per_run[i] for i in top_idx]

print("The best 50 models ：")
for run, recall in zip(top_runs, arr[top_idx]):
    print(f"Run {run} : Recall={recall:.4f}")

In [ ]:
save_dir = "./model"
os.makedirs(save_dir, exist_ok=True)

for i, model in enumerate(top_models, start=1):
    model_path = os.path.join(save_dir, f"model_{i}.cbm")
    model.save_model(model_path, format="cbm")

In [ ]:
arr = np.array(recall_per_run)
top_idx = arr.argsort()[-50:][::-1]

loss_dict = {}

for idx in top_idx:
    model = model_per_run[idx]
    evals_result = model.get_evals_result()
    losses = evals_result["learn"].get(LOSS_FUNCTION, [])
    col_name = f"run_{idx+1}_loss"
    loss_dict[col_name] = losses

loss_df = pd.DataFrame(loss_dict)
loss_df.insert(0, "iteration", range(1, len(loss_df)+1))

In [ ]:
loss_df

In [ ]:
mp_predict = negative.copy()

for i, model in enumerate(top_models, start=1):
    col_name = f"model_{i}_score"
    mp_predict[col_name] = model.predict_proba(negative[pod_features])[:, 1]
    
model_score_cols = [f"model_{i}_score" for i in range(1, len(top_models)+1)]
mp_predict["avg_score"] = mp_predict[model_score_cols].mean(axis=1)
mp_predict["std_score"] = mp_predict[model_score_cols].std(axis=1)
mp_predict = mp_predict.sort_values(by="avg_score", ascending=False)

In [ ]:
mp_predict.to_csv('./data/predict_materials_project.csv', index=False)

In [ ]:
def contains_transition_metal(formula: str) -> bool:
    comp = Composition(formula)
    for el in comp.elements:
        if Element(el).is_transition_metal:
            return True
    return False

mp_predict["has_TM"] = mp_predict["formula"].apply(contains_transition_metal)

bins = np.linspace(0, 1, 11)
labels = [f"{bins[i]:.1f} - {bins[i+1]:.1f}" for i in range(len(bins)-1)]
mp_predict["avg_bin"] = pd.cut(
    mp_predict["avg_score"].clip(0,1),
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True
)

bin_stats = (
    mp_predict
    .groupby("avg_bin")
    .agg(total=("formula", "count"),
         tm_count=("has_TM", "sum"))
)
bin_stats["tm_ratio"] = bin_stats["tm_count"] / bin_stats["total"]

In [ ]:
bin_stats

In [ ]:
mp_predict.has_TM.mean()